## 10.09 大规模预训练 Transformer


### 练习 10.9.1

**题目：** 是否可以使用由不同任务组成的小批量来微调 T5？为什么可以或不可以？GPT-2 呢？

**解答：**

- **T5 可以**。T5 将不同 NLP 任务统一建模为文本到文本（text-to-text）问题：每个任务的输入都拼上任务前缀（如 `translate English to German: ...`、`summarize: ...`），输出统一为文本。因此不同任务的样本天然共享同一套输入输出格式（token 序列），可以混合在一个小批量中，用同一个序列到序列目标（teacher forcing 的交叉熵）微调，无需特殊处理。
- **GPT-2 也可以，但有前提**。GPT-2 是自回归语言模型，只做单向语言建模（给定上文预测下一个词元）。不同任务都可以转成语言建模形式（如把指令和答案拼成一个序列，监督位置是答案部分的词元），因此形式上可以混合训练。但 GPT-2 没有像 T5 那样的任务前缀机制，任务区分完全靠提示（prompt）本身；混合任务时模型需要从 prompt 中自行识别任务，效果不如 T5 直接。
- **主要区别**：T5 的 encoder-decoder 结构对每个任务天然输出完整的生成序列；GPT-2 是纯 decoder，混合小批量微调时所有样本必须同样地按自回归目标计算，若任务结构差异大（如提取式问答与生成式摘要），批内会出现目标位置不一致的问题，需要把每个样本的目标部分单独标识，实现上更麻烦。

### 练习 10.9.2

**题目：** 给定一个强大的语言模型，你能想到哪些应用？

**解答：**

1. **文本生成**：文章/故事/剧本创作、代码生成（补全、注释、翻译代码）。
2. **机器翻译与改写**：多语翻译、风格改写、润色、摘要生成。
3. **问答与对话**：开放域问答、客服对话、教育辅导（讲解知识点）。
4. **信息抽取与结构化**：从文档抽取实体/关系、生成知识图谱、情感分析。
5. **辅助工具**：拼写/语法纠错、检索增强（RAG）的知识库问答、智能编程助手。
6. **多模态接口**：作为视觉-语言模型（ViT + LLM）的语言底座，做图像描述、视觉问答、文档理解。

### 练习 10.9.3

**题目：** 假设要求你通过添加额外层来微调一个语言模型以执行文本分类。你会把层加在哪里？为什么？

**解答：**

- **加在模型尾部（输出端）**：在最后一层 Transformer 表示之上接一个分类头（如对 `[CLS]` 位置或序列池化表示接 `LayerNorm + 线性分类层`，或接一个多层感知机）。
- **原因**：
  1. 预训练模型已经学到丰富的通用语言表示，分类任务只需要把高层的语义表示映射到类别空间，加在尾部参数量小、改动少，可以尽量保留预训练知识；
  2. 若把新层加在中间或底部，会破坏预训练层间的残差结构与表示分布，微调时需要重新适配所有层，容易遗忘预训练能力（灾难性遗忘）且收敛更慢；
  3. 微调时通常配合低学习率或冻结部分底层，只更新头部与顶层，避免过拟合小规模分类数据。

### 练习 10.9.4

**题目：** 考虑序列到序列问题（例如机器翻译），其中输入序列在整个目标序列预测期间始终可用。使用仅解码器 Transformer 建模可能有什么局限性？为什么？

**解答：**

1. **缺少对源序列的双向编码**：仅解码器 Transformer 只能做单向（causal）自注意力，每个目标词元只能看到源序列中它之前的词元（或通过拼接后的位置顺序），无法像 encoder-decoder 的编码器那样同时利用源序列的完整上下文（前后文）。对于机器翻译等任务，理解一个源词往往依赖整句语境，单向编码会损失信息。
2. **需要把源序列与目标序列拼接**：仅解码器建模序列到序列任务时，通常把 `[源序列] + [分隔符] + [目标序列]` 拼成一个序列做自回归生成。源与目标之间是软性注意（attention 权重由模型学），位置编码与注意力模式可能与 encoder-decoder 中的显式交叉注意力（decoder 的 key/value 固定来自编码器输出）不同，对齐能力较弱。
3. **训练与推理代价**：每个目标词元的生成都要对整个拼接序列做注意力（自注意力复杂度 O((n+m)^2)），而 encoder-decoder 中源序列的编码只需计算一次，解码时只需目标端的自注意力加交叉注意力，计算与显存开销更小。
4. **可并行性**：训练时尽管 teacher forcing 可以并行，但推理时仍逐词元生成；且由于输入始终可用（非纯自回归场景），仅解码器结构没有利用这一点做高效的并行编码。

**结论**：仅解码器 Transformer 更适合纯自回归生成（语言模型）；序列到序列任务在源序列完整可用时，encoder-decoder 结构的显式编码 + 交叉注意力通常更强、更高效。

---

## 参考答案来源
参考答案和 PyTorch 代码实现来源：https://datawhalechina.github.io/d2l-ai-solutions-manual/#/